# Fine Tuning

We are gonna finetune the resnet50 model in this notebook.

In [1]:
import torch
from torch import nn
from torch import optim

from torchvision import transforms, datasets
import torchvision
from torchinfo import summary

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

Let's copy the create_resnet50() function from the last notebook and get started.

In [3]:
def create_resnet50(num_classes: int=37):
    weights = torchvision.models.ResNet50_Weights.DEFAULT
    transform = weights.transforms()
    model = torchvision.models.resnet50(weights=weights)

    for param in model.parameters():
        param.requires_grad = False

    model.fc = nn.Linear(in_features=2048, out_features=num_classes, bias=True)

    return model, transform

In [4]:
resnet50, resnet50_transforms = create_resnet50()
resnet50 = resnet50.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 174MB/s]


Instead of using the same transform for on both train and test data, this time we'll use a manual_transform on the train_data. But before that, we are gonna use the same transform to get the previous 91% accuracy because I forgot to save the model earlier.

In [5]:
train_data = datasets.OxfordIIITPet(root='data', split='trainval', transform=resnet50_transforms, download=True)
test_data = datasets.OxfordIIITPet(root='data', split='test', transform=resnet50_transforms, download=True)

100%|██████████| 792M/792M [01:43<00:00, 7.62MB/s]
100%|██████████| 19.2M/19.2M [00:02<00:00, 8.21MB/s]


In [6]:
from going_modular import data_setup
resnet50_train_dataloader, resnet50_test_dataloader = data_setup.create_dataloaders(train_data=train_data,
                                                                  test_data=test_data,
                                                                  batch_size=32)

In [7]:
loss_fn = nn.CrossEntropyLoss()

In [8]:
from going_modular import engine, utils
torch.manual_seed(42)

optimizer = optim.Adam(resnet50.parameters(), lr=0.001)

writer = utils.create_writer(experiment_name="cats_vs_dogs_but_37_ways", model_name="resnet50_redo")
resnet50_results = engine.train(model=resnet50,
                                train_dataloader=resnet50_train_dataloader,
                                test_dataloader=resnet50_test_dataloader,
                                optimizer=optimizer,
                                loss_fn=loss_fn,
                                epochs=10,
                                device=device,
                                writer=writer)

utils.save_model(model=resnet50, model_name="resnet50_10_epochs_frozen_head.pth", target_dir="models")

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1 || Train Loss: 1.87 || Train Accuracy: 68.34 || Test Loss: 1.04 || Test Accuracy: 85.27
Epoch: 2 || Train Loss: 0.56 || Train Accuracy: 93.07 || Test Loss: 0.61 || Test Accuracy: 88.86
Epoch: 3 || Train Loss: 0.34 || Train Accuracy: 95.14 || Test Loss: 0.49 || Test Accuracy: 89.25
Epoch: 4 || Train Loss: 0.23 || Train Accuracy: 96.55 || Test Loss: 0.42 || Test Accuracy: 89.80
Epoch: 5 || Train Loss: 0.19 || Train Accuracy: 97.12 || Test Loss: 0.38 || Test Accuracy: 90.26
Epoch: 6 || Train Loss: 0.15 || Train Accuracy: 98.18 || Test Loss: 0.35 || Test Accuracy: 90.54
Epoch: 7 || Train Loss: 0.12 || Train Accuracy: 98.75 || Test Loss: 0.33 || Test Accuracy: 90.69
Epoch: 8 || Train Loss: 0.10 || Train Accuracy: 98.86 || Test Loss: 0.33 || Test Accuracy: 90.15
Epoch: 9 || Train Loss: 0.09 || Train Accuracy: 99.13 || Test Loss: 0.31 || Test Accuracy: 90.77
Epoch: 10 || Train Loss: 0.07 || Train Accuracy: 99.35 || Test Loss: 0.31 || Test Accuracy: 90.96
Saving the model..
Model has 

You can see the train and test accuracy gap, the model might be overfitting on the training data. We'll do some experiments and try to fix it.

# Experiment # 1

Let's move further with more changes and experiments. As I said earlier, I will be using a manual transform on the train data...but even before that, let's do another experiment.

In [9]:
resnet50_v2, _ = create_resnet50()
resnet50_v2

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

You can see that there are 4 layers in the model, all frozen(in the create_resnet50 function). Let's unfreeze the last layer and train the model to see the if it improves.

In [10]:
from going_modular import utils
resnet50_v2 = utils.load_model(resnet50_v2, "models/resnet50_10_epochs_frozen_head.pth", device=device)

In [11]:
for param in resnet50_v2.layer4.parameters():
  param.requires_grad = True

In [12]:
summary(model=resnet50_v2,
                  input_size=[32, 3, 224, 224],
                  col_names=['input_size', 'output_size', 'trainable'],
                  row_settings=['var_names'])

Layer (type (var_name))                  Input Shape               Output Shape              Trainable
ResNet (ResNet)                          [32, 3, 224, 224]         [32, 37]                  Partial
├─Conv2d (conv1)                         [32, 3, 224, 224]         [32, 64, 112, 112]        False
├─BatchNorm2d (bn1)                      [32, 64, 112, 112]        [32, 64, 112, 112]        False
├─ReLU (relu)                            [32, 64, 112, 112]        [32, 64, 112, 112]        --
├─MaxPool2d (maxpool)                    [32, 64, 112, 112]        [32, 64, 56, 56]          --
├─Sequential (layer1)                    [32, 64, 56, 56]          [32, 256, 56, 56]         False
│    └─Bottleneck (0)                    [32, 64, 56, 56]          [32, 256, 56, 56]         False
│    │    └─Conv2d (conv1)               [32, 64, 56, 56]          [32, 64, 56, 56]          False
│    │    └─BatchNorm2d (bn1)            [32, 64, 56, 56]          [32, 64, 56, 56]          False
│    │    

You can see that the layer4 parameters are now trainable. Let's train the model.

For optimizer, we will be using two different learning rates. As the layer4 already holds pretrained values, we'll decrease its learning rate so it learns in small steps. And for the fc layer, we'll keep the same learning rate.

In [13]:
torch.manual_seed(42)

optimizer = optim.Adam([
    {"params": resnet50_v2.layer4.parameters(), "lr": 0.0001},
    {"params": resnet50_v2.fc.parameters(),     "lr": 0.001},
])

writer = utils.create_writer(experiment_name="cats_vs_dogs_but_37_ways", model_name="resnet_50_unfrozen_layer4")
resnet50_v2_results = engine.train(model=resnet50_v2,
                                train_dataloader=resnet50_train_dataloader,
                                test_dataloader=resnet50_test_dataloader,
                                optimizer=optimizer,
                                loss_fn=loss_fn,
                                epochs=5,
                                device=device,
                                writer=writer)

utils.save_model(model=resnet50_v2, model_name="resnet50_unfrozen_layer4.pth", target_dir="models")

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 || Train Loss: 0.06 || Train Accuracy: 98.59 || Test Loss: 0.32 || Test Accuracy: 90.76
Epoch: 2 || Train Loss: 0.02 || Train Accuracy: 99.59 || Test Loss: 0.31 || Test Accuracy: 90.71
Epoch: 3 || Train Loss: 0.01 || Train Accuracy: 99.86 || Test Loss: 0.30 || Test Accuracy: 91.25
Epoch: 4 || Train Loss: 0.01 || Train Accuracy: 99.86 || Test Loss: 0.31 || Test Accuracy: 90.98
Epoch: 5 || Train Loss: 0.01 || Train Accuracy: 99.84 || Test Loss: 0.33 || Test Accuracy: 91.29
Saving the model..
Model has been saved successfully.


The model still seems to be overfitting on the training data, and this time it seems worse than the last time because the train/test loss gap is pretty big.(0.01/0.33)

# Experiment # 2

Now, we are gonna create a manual transform for the next experiment's training. But we will use the same tranform as now for testing.

Also, we will try different learning rates.

In [14]:
resnet50_v3, _ = create_resnet50()
resnet50_v3 = utils.load_model(resnet50_v3, "models/resnet50_10_epochs_frozen_head.pth", device=device)

In [15]:
for param in resnet50_v3.layer4.parameters():
  param.requires_grad = True

In [16]:
resnet50_transforms

ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [17]:
manual_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [18]:
train_data_augmented = datasets.OxfordIIITPet(root="data", split="trainval",
                                        transform=manual_train_transform, download=True)

train_dataloader_augmented, _ = data_setup.create_dataloaders(train_data=train_data_augmented,
                                                test_data=test_data,
                                                batch_size=32)

In [19]:
torch.manual_seed(42)

optimizer = optim.Adam([
    {"params": resnet50_v3.layer4.parameters(), "lr": 0.0001},
    {"params": resnet50_v3.fc.parameters(),     "lr": 0.001},
])

writer = utils.create_writer(experiment_name="cats_vs_dogs_but_37_ways", model_name="resnet_50_unfrozen_layer4_and_manual_transform")
resnet50_v3_results = engine.train(model=resnet50_v3,
                                train_dataloader=train_dataloader_augmented,
                                test_dataloader=resnet50_test_dataloader,
                                optimizer=optimizer,
                                loss_fn=loss_fn,
                                epochs=5,
                                device=device,
                                writer=writer)

utils.save_model(model=resnet50_v3, model_name="resnet_50_unfrozen_layer4_and_manual_transform.pth", target_dir="models")

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 || Train Loss: 0.48 || Train Accuracy: 85.73 || Test Loss: 0.29 || Test Accuracy: 90.57
Epoch: 2 || Train Loss: 0.37 || Train Accuracy: 88.64 || Test Loss: 0.26 || Test Accuracy: 91.88
Epoch: 3 || Train Loss: 0.32 || Train Accuracy: 90.19 || Test Loss: 0.26 || Test Accuracy: 91.83
Epoch: 4 || Train Loss: 0.30 || Train Accuracy: 90.87 || Test Loss: 0.27 || Test Accuracy: 91.18
Epoch: 5 || Train Loss: 0.28 || Train Accuracy: 91.44 || Test Loss: 0.30 || Test Accuracy: 90.88
Saving the model..
Model has been saved successfully.


Now the overfitting problem is seemed to be solved. The model is generalizing well on the training data.

Let's take a look at tensorboard.

In [2]:
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 52326), started 0:00:11 ago. (Use '!kill 52326' to kill it.)

You can see that for our two experiments, the model with manual transform has somewhat identical train and test loss(0.2799 for training, and 0.3019 for testing) but for the first experiment, the training loss is really low which indicates overfitting(0.0088 for training and 0.3309 for testing).

The experiment 2 model seems to be generalizing well. SO we'll continue with it.